# DeepCache — Stable Diffusion 1.5 no Google Colab

Notebook para comparação Text-to-Image:
1. Stable Diffusion sem cache;
2. DeepCache oficial (`cache_interval=3`, `cache_branch_id=0`);
3. DeepCache adaptativo por variação relativa L2 dos latents.

Experimento: 10 prompts, seed 42, 50 passos DDIM e resolução 512×512.

## Bloco 1 — Ambiente e Hardware

In [ ]:
# Instale dependências sem reinstalar o PyTorch/CUDA já configurado pelo Colab.
%pip install -q "diffusers>=0.43,<0.44" "transformers>=5.24,<6" "huggingface-hub>=1.32,<2" "tokenizers>=0.23.1,<0.24" "sentence-transformers>=6.3,<7" "accelerate>=1.12" "safetensors>=0.7" jedi lpips torchmetrics DeepCache

print("Instalação concluída. Reinicie o runtime antes de executar a próxima célula.")
print("Colab: Ambiente de execução > Reiniciar sessão.")

In [ ]:
# Execute após reiniciar o runtime do Colab.
import importlib.metadata
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Diffusers:", importlib.metadata.version("diffusers"))
print("Transformers:", importlib.metadata.version("transformers"))
print("Hugging Face Hub:", importlib.metadata.version("huggingface-hub"))
print("Tokenizers:", importlib.metadata.version("tokenizers"))
print("CUDA disponível:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Ative uma GPU NVIDIA no Google Colab: Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("GPU:", gpu_name)
print(f"VRAM total: {vram_gb:.2f} GB")

if vram_gb < 14:
    print("AVISO: use resolução menor ou uma GPU com pelo menos 15 GB se ocorrer OOM.")

!pip check

## Bloco 2 — Baseline e DeepCache Oficial

In [ ]:
from pathlib import Path
import json
import time
import requests

import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler
from DeepCache import DeepCacheSDHelper
from PIL import Image, ImageDraw

MODEL_ID = "runwayml/stable-diffusion-v1-5"
SEED = 42
STEPS = 50
GUIDANCE = 7.5
WIDTH = 512
HEIGHT = 512
TAU = 0.020

project_dir = Path("/content/deepcache-experiment")
results_dir = project_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)

prompts_file = project_dir / "data" / "prompts_geracao.txt"
if prompts_file.exists():
    prompts = [line.strip() for line in prompts_file.read_text(encoding="utf-8").splitlines() if line.strip()]
else:
    prompts = []

additional_prompts = [
    "A red dragon flying over a medieval castle at sunrise, fantasy concept art",
    "A modern glass house in a green forest, architectural photography",
    "A close-up portrait of an astronaut woman reflected in her helmet visor",
    "A steaming cup of coffee beside an open book in a cozy library",
    "An underwater coral reef with colorful tropical fish, natural documentary photography",
]

prompts = (prompts + additional_prompts)[:10]
if len(prompts) < 10:
    prompts += [f"A detailed cinematic digital illustration of a landscape, variation {i+1}" for i in range(10-len(prompts))]

print(f"Quantidade de prompts: {len(prompts)}")

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
).to("cuda")
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.enable_attention_slicing("max")

def reset_seed(seed=SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def generate(prompt, mode):
    reset_seed()
    generator = torch.Generator(device="cuda").manual_seed(SEED)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.perf_counter()
    image = pipe(
        prompt=prompt,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=WIDTH,
        height=HEIGHT,
        generator=generator,
        output_type="pil",
    ).images[0]
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return image, elapsed, peak_mb

baseline_images = []
official_images = []
baseline_times = []
official_times = []
baseline_vram = []
official_vram = []

official_helper = DeepCacheSDHelper(pipe=pipe)
official_helper.set_params(cache_interval=3, cache_branch_id=0)

for i, prompt in enumerate(prompts, start=1):
    image, elapsed, peak = generate(prompt, "baseline")
    baseline_images.append(image)
    baseline_times.append(elapsed)
    baseline_vram.append(peak)
    image.save(results_dir / f"prompt_{i:02d}_baseline.png")

    print(f"{i:02d}/{len(prompts)} | baseline={baseline_times[-1]:.2f}s")

for i, prompt in enumerate(prompts, start=1):
    official_helper.enable()
    try:
        image, elapsed, peak = generate(prompt, "deepcache_official")
    finally:
        official_helper.disable()
    official_images.append(image)
    official_times.append(elapsed)
    official_vram.append(peak)
    image.save(results_dir / f"prompt_{i:02d}_deepcache_oficial.png")

    print(f"{i:02d}/{len(prompts)} | DeepCache oficial={official_times[-1]:.2f}s")

## Bloco 3 — Intervenção Técnica: Cache Adaptativo

In [ ]:
import torch

adaptive_helper = DeepCacheSDHelper(pipe=pipe)
adaptive_helper.set_params(cache_interval=STEPS + 1, cache_branch_id=0)
original_is_skip_step = adaptive_helper.is_skip_step
adaptive_policy = {"force_full_unet": True}

def adaptive_is_skip_step(block_i, layer_i, blocktype="down"):
    if adaptive_policy["force_full_unet"]:
        return False
    return original_is_skip_step(block_i, layer_i, blocktype)

adaptive_helper.is_skip_step = adaptive_is_skip_step

adaptive_reference_latent = None
adaptive_update_count = 0

def adaptive_step_callback(pipe, step, timestep, callback_kwargs):
    global adaptive_reference_latent, adaptive_update_count
    z_t = callback_kwargs["latents"].detach()
    if adaptive_reference_latent is None:
        relative_l2 = float("inf")
    else:
        z_cur = z_t.float()
        z_ref = adaptive_reference_latent.float()
        relative_l2 = (
            torch.linalg.vector_norm(z_cur - z_ref)
            / (torch.linalg.vector_norm(z_ref) + 1e-8)
        ).item()

    adaptive_policy["force_full_unet"] = relative_l2 > TAU
    if adaptive_policy["force_full_unet"]:
        adaptive_reference_latent = z_t.clone()
        adaptive_update_count += 1
    return callback_kwargs

adaptive_images = []
adaptive_times = []
adaptive_vram = []
adaptive_update_counts = []

for i, prompt in enumerate(prompts, start=1):
    adaptive_reference_latent = None
    adaptive_update_count = 1
    adaptive_policy["force_full_unet"] = True
    adaptive_helper.enable()
    try:
        reset_seed()
        generator = torch.Generator(device="cuda").manual_seed(SEED)
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        start = time.perf_counter()
        image = pipe(
            prompt=prompt,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            width=WIDTH,
            height=HEIGHT,
            generator=generator,
            callback_on_step_end=adaptive_step_callback,
            callback_on_step_end_tensor_inputs=["latents"],
            output_type="pil",
        ).images[0]
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
    finally:
        adaptive_helper.disable()

    adaptive_images.append(image)
    adaptive_times.append(elapsed)
    adaptive_vram.append(peak)
    adaptive_update_counts.append(adaptive_update_count)
    image.save(results_dir / f"prompt_{i:02d}_deepcache_adaptativo.png")

    print(f"{i:02d}/{len(prompts)} | adaptativo={elapsed:.2f}s | atualizações completas={adaptive_update_count}")

## Bloco 4 — Avaliação e Exportação

In [ ]:
import gc
import numpy as np
import lpips
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from transformers import CLIPModel, CLIPProcessor

# Libera a GPU antes de carregar CLIP e LPIPS.
del pipe
gc.collect()
torch.cuda.empty_cache()

lpips_metric = lpips.LPIPS(net="alex").eval().cpu()

def pil_to_lpips_tensor(image):
    array = np.asarray(image.resize((256, 256))).astype(np.float32) / 127.5 - 1.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)
    return tensor

def lpips_vs_baseline(baseline, candidate):
    with torch.no_grad():
        return lpips_metric(
            pil_to_lpips_tensor(baseline),
            pil_to_lpips_tensor(candidate),
        ).item()

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval().cpu()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def clip_score(prompt, image):
    inputs = clip_processor(text=[prompt], images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = clip_model(**inputs)
        image_features = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
        text_features = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
        return (image_features @ text_features.T).item()

official_lpips = []
adaptive_lpips = []
baseline_clip = []
official_clip = []
adaptive_clip = []

for prompt, baseline, official, adaptive in zip(
    prompts, baseline_images, official_images, adaptive_images
):
    official_lpips.append(lpips_vs_baseline(baseline, official))
    adaptive_lpips.append(lpips_vs_baseline(baseline, adaptive))
    baseline_clip.append(clip_score(prompt, baseline))
    official_clip.append(clip_score(prompt, official))
    adaptive_clip.append(clip_score(prompt, adaptive))

baseline_mean_time = float(np.mean(baseline_times))
official_mean_time = float(np.mean(official_times))
adaptive_mean_time = float(np.mean(adaptive_times))

baseline_peak = float(max(baseline_vram))
official_peak = float(max(official_vram))
adaptive_peak = float(max(adaptive_vram))

summary = [
    {
        "Método": "Stable Diffusion sem cache",
        "Latência média (s)": baseline_mean_time,
        "Speedup": 1.0,
        "Pico VRAM (MB)": baseline_peak,
        "CLIP Score médio": float(np.mean(baseline_clip)),
        "LPIPS médio vs. sem cache": 0.0,
    },
    {
        "Método": "DeepCache oficial",
        "Latência média (s)": official_mean_time,
        "Speedup": baseline_mean_time / official_mean_time,
        "Pico VRAM (MB)": official_peak,
        "CLIP Score médio": float(np.mean(official_clip)),
        "LPIPS médio vs. sem cache": float(np.mean(official_lpips)),
    },
    {
        "Método": "DeepCache adaptativo",
        "Latência média (s)": adaptive_mean_time,
        "Speedup": baseline_mean_time / adaptive_mean_time,
        "Pico VRAM (MB)": adaptive_peak,
        "CLIP Score médio": float(np.mean(adaptive_clip)),
        "LPIPS médio vs. sem cache": float(np.mean(adaptive_lpips)),
    },
]

rows = [
    "| Método | Latência média (s) | Speedup | Pico VRAM (MB) | CLIP Score médio | LPIPS médio vs. sem cache |",
    "|---|---:|---:|---:|---:|---:|",
]
for row in summary:
    rows.append(
        f"| {row['Método']} | {row['Latência média (s)']:.3f} | {row['Speedup']:.2f}x | "
        f"{row['Pico VRAM (MB)']:.1f} | {row['CLIP Score médio']:.4f} | "
        f"{row['LPIPS médio vs. sem cache']:.4f} |"
    )

print("\nTabela final para o relatório:\n")
print("\n".join(rows))

results_csv = results_dir / "metricas_resultados.csv"
import csv
with results_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "prompt",
        "baseline_time_s", "official_time_s", "adaptive_time_s",
        "baseline_peak_vram_mb", "official_peak_vram_mb", "adaptive_peak_vram_mb",
        "baseline_clip", "official_clip", "adaptive_clip",
        "official_lpips_vs_baseline", "adaptive_lpips_vs_baseline",
        "adaptive_update_count",
    ])
    for i, prompt in enumerate(prompts):
        writer.writerow([
            prompt,
            baseline_times[i], official_times[i], adaptive_times[i],
            baseline_vram[i], official_vram[i], adaptive_vram[i],
            baseline_clip[i], official_clip[i], adaptive_clip[i],
            official_lpips[i], adaptive_lpips[i], adaptive_update_counts[i],
        ])

(results_dir / "resumo_resultados.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

thumb_w, thumb_h = 256, 256
canvas = Image.new(
    "RGB",
    (thumb_w * 3, (thumb_h + 90) * len(prompts)),
    "white",
)
draw = ImageDraw.Draw(canvas)

for i, prompt in enumerate(prompts):
    y = i * (thumb_h + 90)
    draw.text((10, y + 8), f"Prompt {i+1}: {prompt[:95]}", fill="black")
    draw.text((10, y + 48), f"Sem cache: {baseline_times[i]:.2f}s", fill="black")
    draw.text((thumb_w + 10, y + 48), f"DeepCache: {official_times[i]:.2f}s", fill="black")
    draw.text((2 * thumb_w + 10, y + 48), f"Adaptativo: {adaptive_times[i]:.2f}s", fill="black")
    canvas.paste(baseline_images[i].resize((thumb_w, thumb_h)), (0, y + 90))
    canvas.paste(official_images[i].resize((thumb_w, thumb_h)), (thumb_w, y + 90))
    canvas.paste(adaptive_images[i].resize((thumb_w, thumb_h)), (2 * thumb_w, y + 90))

grid_path = results_dir / "comparacao_visual.png"
canvas.save(grid_path)
print("Grade visual:", grid_path)
print("CSV:", results_csv)
display(canvas)